# Migrant Archive — Colab Transcription

Transcribe a YouTube video using **whisperx large-v3** on Colab's free T4 GPU.

### What this notebook does
1. Clones the project repo (or uses uploaded files)
2. Downloads audio from YouTube via `yt-dlp`
3. Transcribes using `whisperx` (large-v3, GPU)
4. Saves the result as JSON to your Google Drive

### Estimated time
| Video length | Transcription time on T4 GPU |
|-------------|------------------------------|
| 5 min | ~15 seconds |
| 30 min | ~2 minutes |
| 2 hours | ~8-10 minutes |

### After this notebook
Download the JSON from Drive → place in `data/raw/whisper/` → run `rag_test.py --rebuild`

---
> For speaker labels, set a HuggingFace token in section 5.
---

## 1. Install Dependencies

In [1]:
!pip install -q yt-dlp whisperx
!apt-get update -qq && apt-get install -y -qq nodejs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 139.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [10]:
!pip install "transformers==4.48.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 106.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


## 2. Mount Google Drive

This is where the JSON output will be saved. You'll need to authorize access.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/migrant-archive"
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/audio", exist_ok=True)
LOCAL_AUDIO_DIR = "/content/audio"
os.makedirs("/content/audio", exist_ok=True)

print(f"Drive mounted. Output → {DRIVE_ROOT}/output/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Output → /content/drive/MyDrive/migrant-archive/output/


## 3. Get the Ingestion Code

Choose **one** method below. Option A (git clone) is recommended — it pulls all files with correct imports.

### Option A: Clone the repo (recommended)

In [2]:
import sys
from pathlib import Path

# Clone the repo
REPO_URL = "https://github.com/Sebastianlopez-dev/migrant-archive.git"
!git clone {REPO_URL} /content/migrant-archive

# Add backend/core to Python path
sys.path.insert(0, str(Path("/content/migrant-archive/backend/core")))

print("Repo cloned and path configured")

Cloning into '/content/migrant-archive'...
fatal: could not read Username for 'https://github.com': No such device or address
Repo cloned and path configured


### Option B: Upload files manually

If you prefer not to clone, upload these 3 files from `backend/core/`:
- `ingestion.py`
- `ingestion_audio.py`  
- `ingestion_colab.py`

Use the Files panel (left sidebar) to upload, then run this cell:

In [3]:
# Run if using Option B (manual upload)
import sys
from pathlib import Path
sys.path.insert(0, "/content")
print("Manual upload path configured")

Manual upload path configured


## 4. Configure Your Video

Set the YouTube URL and language below. For the FILMIG channel, use `es`.
For Catalan/Spanish code-switching, try `ca` or `es`.

In [4]:
# ══════════════════════════════════════════
# EDIT THESE TWO VALUES
# ══════════════════════════════════════════

#VIDEO_URL = "https://www.youtube.com/watch?v=PASTE_VIDEO_ID_HERE"
VIDEO_URL = "https://youtu.be/APgxfNssxGQ"
LANGUAGE  = "es"       # es, en, ca, or "" for auto-detect

# ══════════════════════════════════════════
# Advanced (usually leave as-is)
# ══════════════════════════════════════════

MODEL     = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE    = "cuda"      # cuda (GPU) or cpu

print(f"Video: {VIDEO_URL}")
print(f"Language: {LANGUAGE}  |  Model: {MODEL}  |  Device: {DEVICE}")

Video: https://youtu.be/APgxfNssxGQ
Language: es  |  Model: large-v3  |  Device: cuda


## 5. HuggingFace Token (WhisperX)

WhisperX needs a HuggingFace token to download the speaker diarisation model
(pyannote/speaker-diarization-3.1).

1. Create a token at https://huggingface.co/settings/tokens (read-only is enough)
2. Accept terms at https://hf.co/pyannote/speaker-diarization-3.1
3. Run this cell to set it before transcription

In [5]:
import os
#os.environ["HF_TOKEN"] = "hf_tu_token_aqui"
os.environ["HF_TOKEN"] = ""
# Verify it's set
if os.environ.get("HF_TOKEN", "").startswith("hf_"):
    print("HF_TOKEN ready for WhisperX diarisation")
else:
    print("... HF_TOKEN not set — transcription will work but speakers won't be labelled")

HF_TOKEN ready for WhisperX diarisation


## 6. YouTube Authentication (cookies)

YouTube may block Colab's datacenter IP and ask for authentication.
If you see `Sign in to confirm you're not a bot`, follow these steps.

#### On your local machine (Mac terminal)
```bash
# Export cookies from Brave. Use 'chrome' if you use Chrome.
yt-dlp --cookies-from-browser brave "https://www.youtube.com" \
       --cookies cookies.txt --skip-download
```

In Colab
1. Upload cookies.txt using the Files panel (left sidebar — folder icon)
2. Run the cell below to inject cookies into all yt-dlp calls
3. Then run Section 7 (Run Transcription) as normal

> Skip this section if you don't get a bot-detection error.
If the video downloads fine without cookies, you don't need this.

In [6]:
import ingestion
import yt_dlp
from pathlib import Path

COOKIE_FILE = "/content/cookies.txt"

# Replace _fetch_metadata with cookie-aware version
def _fetch_metadata_cookies(video_url: str) -> dict:
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        return ydl.extract_info(video_url, download=False)

# Replace _download_audio with cookie-aware version
def _download_audio_cookies(video_url: str, output_dir: str = "data/audio") -> Path:
    audio_dir = Path(output_dir)
    audio_dir.mkdir(parents=True, exist_ok=True)

    info = _fetch_metadata_cookies(video_url)
    audio_path = audio_dir / f"{info['id']}.mp3"

    if audio_path.exists():
        return audio_path

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": str(audio_dir / "%K(id)s.%(ext)s"),
        "noplaylist": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "quiet": True,
        "no_warnings": True,
        "cookiefile": COOKIE_FILE,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(video_url, download=True)

    return audio_path

# Monkeypatch the module
ingestion._fetch_metadata = _fetch_metadata_cookies
ingestion._download_audio = _download_audio_cookies

print("Cookies injected — yt-dlp will authenticate as you")

Cookies injected — yt-dlp will authenticate as you


In [15]:
print("Reinstalling numpy and scipy to resolve ImportErrors...")
!pip uninstall -y numpy scipy
!pip install numpy scipy
print("Numpy and Scipy reinstalled. Please restart the Colab runtime (Runtime > Restart runtime) and then re-run the cells from '1. Install Dependencies' onwards.")

Reinstalling numpy and scipy to resolve ImportErrors...
Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6
Found existing installation: scipy 1.16.3
Uninstalling scipy-1.16.3:
  Successfully uninstalled scipy-1.16.3
  Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
Using cached numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 76.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 

Numpy and Scipy reinstalled. Please restart the Colab runtime (Runtime > Restart runtime) and then re-run the cells from '1. Install Dependencies' onwards.


## 7. Run Transcription

This cell downloads the audio, runs Whisper, and saves the result.
For a 2-hour video, expect ~8-10 minutes on the free T4 GPU.

In [7]:
import time
import json
from ingestion import VideoData, _fetch_metadata, _build_videodata, _download_audio
from ingestion_audio import _detect_device, _compute_type_for, _transcribe_audio

# ── Step 1: Fetch metadata ────────────────────────────────────────────
print("Fetching video metadata ...")
info = _fetch_metadata(VIDEO_URL)
video_id = info["id"]
title = info.get("title", "Unknown")
duration_mins = info.get("duration", 0) / 60

print(f"   Title: {title}")
print(f"   Duration: {duration_mins:.0f} min")
print(f"   Channel: {info.get('channel', 'Unknown')}")

# ── Step 2: Download audio ────────────────────────────────────────────
print(f"\nDownloading audio ...")
t0 = time.time()
audio_path = _download_audio(VIDEO_URL, output_dir=LOCAL_AUDIO_DIR)
print(f"   Done in {time.time() - t0:.0f}s → {audio_path}")

# ── Step 3: Transcribe ────────────────────────────────────────────────
print(f"\nTranscribing with whisperx ({MODEL}, {DEVICE}) ...")
print(f"    This may take a few minutes for long videos ...")
t0 = time.time()

hf_token = os.environ.get("HF_TOKEN") or None
segments = _transcribe_audio(
    audio_path=str(audio_path),
    language=LANGUAGE,
    model_size=MODEL,
    device=DEVICE,
    hf_token=hf_token,
)

elapsed = time.time() - t0
rtf = elapsed / (info.get("duration", 1) / 60)  # real-time factor
print(f"   Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {rtf:.1f}x real-time")
print(f"   Segments: {len(segments)}")

# ── Step 4: Build VideoData & save ────────────────────────────────────
video_data = _build_videodata(info, segments)
output_path = video_data.save_json(output_dir=f"{DRIVE_ROOT}/output")

print(f"\nSaved to Drive: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.0f} KB")

# ── Step 5: Preview ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PREVIEW (first 300 chars):")
print(f"{'─'*60}")
print(video_data.full_text[:100])
print(f"{'─'*60}")

# ── Step 6: Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TRANSCRIPTION COMPLETE")
print(f"{'='*60}")
print(f"Video:    {title}")
print(f"Duration: {duration_mins:.0f} min")
print(f"Segments: {len(segments)}")
print(f"Chars:    {len(video_data.full_text):,}")
print(f"Speed:    {rtf:.1f}x real-time")
print(f"Saved:    {output_path}")
print(f"\n Next: Download this JSON → place in data/raw/whisper/ → rebuild index")

Fetching video metadata ...


   Title: Presentación FILMIG 2024 (Feria Itinerante del Libro Migrante)
   Duration: 4 min
   Channel: Plataforma Cero



   Done in 10s → /content/audio/APgxfNssxGQ.mp3

Transcribing with whisperx (large-v3, cuda) ...
    This may take a few minutes for long videos ...


tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocabulary.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

2026-06-20 14:16:14 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-06-20 14:16:14 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


INFO: Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`
INFO:lightning.pytorch.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../usr/local/lib/python3.12/dist-packages/whisperx/assets/pytorch_model.bin`


RuntimeError: Failed to load audio: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --ena

---

## 8. Batch Processing (Multiple Videos)

Upload `youtube-links.txt` to Colab (Files panel -> upload), then run this cell.
It reads all video URLs from the file and processes them one by one.

**Already-processed videos are skipped** -- add new URLs to the file and
re-run safely without re-transcribing everything.

> Use this for the first bulk run. Use **Section 7** for a single new video.

In [ ]:
import time
import os
from pathlib import Path
from ingestion import _fetch_metadata
from ingestion_audio import extract_single_video

# ══════════════════════════════════════════
# CONFIGURE
# ══════════════════════════════════════════
LINKS_FILE = "/content/youtube-links.txt"   # uploaded via Files panel
LANGUAGE   = "es"
MODEL      = "large-v3"
DEVICE     = "cuda"
OUTPUT_DIR = f"{DRIVE_ROOT}/output"
AUDIO_DIR  = f"{DRIVE_ROOT}/audio"
hf_token   = os.environ.get("HF_TOKEN") or None

# ══════════════════════════════════════════
# Read links file (skip comments)
# ══════════════════════════════════════════
raw = Path(LINKS_FILE).read_text(encoding="utf-8").splitlines()
urls = [line.strip() for line in raw if line.strip() and not line.strip().startswith("#")]

print(f"{len(urls)} video(s) in links file")

# ══════════════════════════════════════════
# Process each video
# ══════════════════════════════════════════
ok, skipped, failed = 0, 0, 0

for i, url in enumerate(urls, 1):
    print(f"\n{'═'*60}")
    print(f"[{i}/{len(urls)}] {url}")
    print(f"{'═'*60}")

    # ── Check if already processed ──
    try:
        info = _fetch_metadata(url)
        video_id = info["id"]
    except Exception as e:
        print(f"   [ERROR] Metadata fetch failed: {e}")
        failed += 1
        continue

    json_path = Path(OUTPUT_DIR) / f"{video_id}.json"
    if json_path.exists():
        print(f"   [SKIP] Already exists: {json_path.name}")
        skipped += 1
        continue

    # ── Transcribe ──
    print(f"   Title: {info.get('title', 'Unknown')}")
    print(f"   Duration: {info.get('duration', 0) / 60:.0f} min")
    print(f"   Transcribing ...")

    t0 = time.time()
    try:
        data = extract_single_video(
            video_url=url,
            languages=[LANGUAGE],
            model_size=MODEL,
            device=DEVICE,
            output_dir=OUTPUT_DIR,
            audio_dir=AUDIO_DIR,
            hf_token=hf_token,
        )
        saved = data.save_json(output_dir=OUTPUT_DIR)
        elapsed = time.time() - t0
        print(f"   [OK] {len(data.transcript_segments)} segments in {elapsed:.0f}s")
        print(f"        Saved: {saved}")
        ok += 1
    except Exception as e:
        print(f"   [ERROR] Failed: {e}")
        failed += 1
    time.sleep(45)

# ══════════════════════════════════════════
# Summary
# ══════════════════════════════════════════

print(f"\n{'='*60}")
print(f"BATCH COMPLETE")
print(f"{'='*60}")
print(f"  [OK]    Processed: {ok}")
print(f"  [SKIP]  Skipped:   {skipped}")
print(f"  [ERROR] Failed:    {failed}")
print(f"\n  Output: {OUTPUT_DIR}/")
print(f"\n  Next: Download JSONs from Drive -> place in data/raw/whisper/ -> rebuild index")


## 9. Download the Result

The JSON is saved in your Google Drive at `migrant-archive/output/{video_id}.json`.

**Option A**: Download directly from [drive.google.com](https://drive.google.com) → My Drive → migrant-archive → output

**Option B**: Run this cell to download via Colab:

In [ ]:
from google.colab import files

# Download the JSON file to your local machine
files.download(str(output_path))

print("\nPlace this file in: migrant-archive/data/raw/whisper/")
print("Then run: python backend/scripts/rag_test.py --rebuild")

---

## 10. Troubleshooting

| Problem | Fix |
|---------|-----|
| `yt-dlp` says video unavailable | Video may be private/age-restricted. Try another URL. |
| Out of memory (OOM) | Switch to `MODEL = "medium"` — still excellent quality, less RAM. |
| GPU not available | Colab T4 quota exhausted. Wait a few hours or use `DEVICE = "cpu"` (much slower). |
| `whisperx` import error | Re-run the install cell
(!pip install). Colab sometimes loses packages. |
| Drive mount fails | Re-run the mount cell. Make sure you're logged into the correct Google account. |